<a href="https://colab.research.google.com/github/yc-115/DSCP/blob/main/%E8%B3%87%E6%96%99%E7%A7%91%E5%AD%B8%E6%9C%9F%E6%9C%AB_%E7%A7%91%E6%8A%80117%E5%8A%89%E7%A5%90%E7%BE%BD%E3%80%81%E5%91%A8%E6%80%A1%E8%BE%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install gradio pandas matplotlib nltk -q

In [ ]:
import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    nltk.download('vader_lexicon')

sia = SentimentIntensityAnalyzer()

try:
    df_yt = pd.read_csv("youtube.csv", encoding="utf-8-sig")
except:
    df_yt = pd.read_csv("youtube.csv", encoding="latin1")

df_yt = df_yt.dropna(subset=['views', 'likes', 'category_id'])

category_map = {
    1: "Film & Animation", 2: "Autos & Vehicles", 10: "Music",
    15: "Pets & Animals", 17: "Sports", 19: "Travel & Events",
    20: "Gaming", 22: "People & Blogs", 23: "Comedy",
    24: "Entertainment", 25: "News & Politics", 26: "Howto & Style",
    27: "Education", 28: "Science & Technology", 29: "Nonprofits & Activism"
}

categories_list = [(category_map.get(cid, f"Category {cid}"), cid) for cid in sorted(df_yt['category_id'].unique())]

def analyze_yt_trending(category_id):
    """Function for Tab 1: YouTube Data Analysis"""
    data = df_yt[df_yt['category_id'] == category_id].copy()
    if len(data) == 0:
        return "No data found.", None, None, None

    data['engagement_rate'] = (data['likes'] / data['views']) * 1000
    top10 = data.sort_values(by='views', ascending=False).head(10)

    avg_views = top10['views'].mean()
    avg_engagement = top10['engagement_rate'].mean()
    category_name = category_map.get(category_id, f"ID {category_id}")

    report = f"""
    ### 📊 Analysis Report: {category_name}
    - **Total Videos in Category**: {len(data):,}
    - **Top 10 Avg Views**: {avg_views:,.0f}
    - **Top 10 Avg Engagement**: {avg_engagement:.2f} (Likes per 1k views)
    - **Most Viewed Video**: {top10['title'].iloc[0] if 'title' in top10.columns else 'N/A'}
    """

    fig1, ax1 = plt.subplots(figsize=(8, 5))
    ax1.scatter(top10['views'], top10['likes'], color='#ff0000', alpha=0.6)
    ax1.set_xlabel("Views")
    ax1.set_ylabel("Likes")
    ax1.set_title(f"Views vs Likes ({category_name})")
    plt.tight_layout()

    fig2, ax2 = plt.subplots(figsize=(8, 5))
    ax2.bar(range(1, 11), top10['views'], color='#4285F4')
    ax2.set_xticks(range(1, 11))
    ax2.set_xlabel("Rank")
    ax2.set_ylabel("Views")
    ax2.set_title("Top 10 Views Ranking")
    plt.tight_layout()

    csv_path = "yt_analysis.csv"
    top10.to_csv(csv_path, index=False, encoding="utf-8-sig")

    return report, fig1, fig2, csv_path

def analyze_lyric_sentiment(lyrics):
    """Function for Tab 2: Lyric Sentiment Analysis"""
    if not lyrics.strip():
        return "Please enter lyrics.", None

    scores = sia.polarity_scores(lyrics)
    compound = scores['compound']

    if compound >= 0.05: status, color = "Positive 😊", "green"
    elif compound <= -0.05: status, color = "Negative 😞", "red"
    else: status, color = "Neutral 😐", "gray"

    report = f"""
    ## 🎵 Lyric Sentiment Result
    - **Overall Tone**: <span style='color:{color}; font-weight:bold;'>{status}</span>
    - **Compound Score**: {compound:.4f}

    - **Positive %**: {scores['pos']*100:.1f}%
    - **Neutral %**: {scores['neu']*100:.1f}%
    - **Negative %**: {scores['neg']*100:.1f}%
    """

    labels = ['Negative', 'Neutral', 'Positive']
    vals = [scores['neg'], scores['neu'], scores['pos']]
    colors = ['#ff9999', '#d9d9d9', '#99ff99']

    final_labels, final_vals, final_colors = [], [], []
    for l, v, c in zip(labels, vals, colors):
        if v > 0:
            final_labels.append(l)
            final_vals.append(v)
            final_colors.append(c)

    fig, ax = plt.subplots(figsize=(6, 6))
    if final_vals:
        ax.pie(final_vals, labels=final_labels, colors=final_colors, autopct='%1.1f%%', startangle=140)
        ax.set_title("Emotional Distribution")
    else:
        ax.text(0.5, 0.5, "No clear emotion detected", ha='center')

    return report, fig

with gr.Blocks(title="YouTube & Lyric Insight Hub") as demo:
    gr.Markdown("# 🚀 YouTube & Lyric Insight Hub")

    with gr.Tabs():
        with gr.TabItem("📈 YouTube Trending Analysis"):
            with gr.Row():
                with gr.Column(scale=1):
                    yt_input = gr.Dropdown(choices=categories_list, label="Video Category", value=categories_list[0][1] if categories_list else None)
                    yt_btn = gr.Button("Analyze Trends", variant="primary")
                    yt_file = gr.File(label="Download Top 10 CSV")
                with gr.Column(scale=2):
                    yt_text = gr.Markdown()
            with gr.Row():
                yt_plot1 = gr.Plot(label="Correlation")
                yt_plot2 = gr.Plot(label="Ranking")

        with gr.TabItem("🎵 Lyric Sentiment"):
            with gr.Row():
                with gr.Column(scale=1):
                    lyric_input = gr.Textbox(lines=12, label="Paste Lyrics Here", placeholder="e.g., 'Yesterday, all my troubles seemed so far away...'")
                    lyric_btn = gr.Button("Analyze Sentiment", variant="primary")
                with gr.Column(scale=1):
                    lyric_text = gr.Markdown()
                    lyric_plot = gr.Plot()

    yt_btn.click(analyze_yt_trending, inputs=yt_input, outputs=[yt_text, yt_plot1, yt_plot2, yt_file])
    lyric_btn.click(analyze_lyric_sentiment, inputs=lyric_input, outputs=[lyric_text, lyric_plot])

demo.launch()

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5adb14f0dd737b01bd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
